In [131]:
import pandas as pd
import geopandas as gpd
import numpy as np

from tqdm import tqdm
from shapely import Polygon, make_valid, geometry

In [132]:
ressource_survey_path = "../../../resources/surveys/edgt_lyon"
cleaned_survey_path = "../../../results/surveys/edgt_lyon"
locations_path = "../../../resources/locations"

output_path = "../../../results/surveys/edgt_lyon/trips.geoparquet"

In [133]:
if "papermill" in locals():
    survey_path = papermill.input["survey"]
    spatial_path = papermill.input["spatial"]

    output_path = papermill.output[0]

In [149]:
# Load spatial data

gdf_housing: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_housing.gpkg" % locations_path, layer="housing")
gdf_work: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_work.gpkg" % locations_path, layer="work")
gdf_education: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_education.gpkg" % locations_path, layer="education")
gdf_secondary: gpd.GeoDataFrame = gpd.read_file("%s/lyon_all_potential_secondary.gpkg" % locations_path, layer="secondary")

gdf_zones: gpd.GeoDataFrame = gpd.read_file("%s/EDGT_AML2015_ZF_GT.TAB" % ressource_survey_path).to_crs("EPSG:2154")

In [150]:
gdf_housing


,home_location_id,weight,iris_id,commune_id,fake,location_id,geometry
0,258495570,1.0,010010000,01001,False,home_0,POINT (848193.38 6563109.52)
1,258495569,1.0,010010000,01001,False,home_1,POINT (848203.98 6563089.67)
2,258495600,1.0,010010000,01001,False,home_2,POINT (848214.41 6563076.71)
3,258495411,1.0,010010000,01001,False,home_3,POINT (848221.98 6562965.48)
4,258495331,0.5,010010000,01001,False,home_4,POINT (848245.17 6562942.05)
...,...,...,...,...,...,...,...
679995,317038059,1.5,422710000,42271,False,home_679995,POINT (822215.27 6489639.04)
679996,317038059,1.5,422710000,42271,False,home_679996,POINT (822220.42 6489631.7)
679997,317038060,1.0,422710000,42271,False,home_679997,POINT (822215.27 6489639.04)
679998,317038114,2.0,422710000,42271,False,home_679998,POINT (822191.53 6489646.46)


In [135]:
df_households = pd.read_parquet("%s/households.parquet" % cleaned_survey_path)
df_households

,edgt_household_id,zone_id,household_id,number_of_cars,number_of_motorbikes,number_of_bicycles
0,10100184,101001,0,0,0,2
1,101001115,101001,1,1,0,1
2,1010022,101002,2,1,0,2
3,1010024,101002,3,0,0,0
4,1010025,101002,4,0,0,0
...,...,...,...,...,...,...
6610,712451647,712451,16356,1,0,0
6611,712451653,712451,16357,1,0,1
6612,712451686,712451,16358,1,0,3
6613,712451742,712451,16359,1,2,0


In [136]:


# Merge df_households with gdf_zones to get the geometries
gdf_zones["zone_id"] = gdf_zones["ZF2015_Nouveau_codage"].astype(int)



# Function to select a random point based on weight
def process_group(household_group):
    zone_id = household_group.name

    group_zone_geometry: Polygon = gdf_zones[gdf_zones["zone_id"] == zone_id]["geometry"].iloc[0]
    # Filter gdf_housing based on whether the point is within the zone geometry
    if not group_zone_geometry.is_valid:
        group_zone_geometry = make_valid(group_zone_geometry)
    
    filtered_housing: gpd.GeoDataFrame = gdf_housing[gdf_housing.within(group_zone_geometry)]
    
    assigned_housing = filtered_housing.sample(n=len(household_group), weights="weight", replace=True)
    assigned_housing = assigned_housing["geometry"].reset_index(drop=True)
    return assigned_housing

if True:

    df_households["geometry"] = df_households.groupby('zone_id').apply(process_group).reset_index(drop=True)

    gdf_households = gpd.GeoDataFrame(df_households, geometry="geometry", crs="EPSG:2154")
    gdf_households.to_parquet("%s/households.geoparquet" % cleaned_survey_path)

else: 
    gdf_households = gpd.read_file("%s/households.geojson" % cleaned_survey_path, crs="EPSG:2154")


C:\Users\lebescond\AppData\Local\Temp\ipykernel_18244\2894887477.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_households["geometry"] = df_households.groupby('zone_id').apply(process_group).reset_index(drop=True)


In [137]:
gdf_households

,edgt_household_id,zone_id,household_id,number_of_cars,number_of_motorbikes,number_of_bicycles,geometry
0,10100184,101001,0,0,0,2,POINT (841431.79 6517432.35)
1,101001115,101001,1,1,0,1,POINT (841626.19 6518317.26)
2,1010022,101002,2,1,0,2,POINT (841636.21 6518038.12)
3,1010024,101002,3,0,0,0,POINT (841516.24 6517746.99)
4,1010025,101002,4,0,0,0,POINT (841251.48 6517663.24)
...,...,...,...,...,...,...,...
6610,712451647,712451,16356,1,0,0,POINT (849640.11 6522368.47)
6611,712451653,712451,16357,1,0,1,POINT (849817.573 6522333.5)
6612,712451686,712451,16358,1,0,3,POINT (849817.573 6522333.5)
6613,712451742,712451,16359,1,2,0,POINT (849463.56 6522520.24)


In [138]:
df_persons: pd.DataFrame = pd.read_parquet("%s/persons.parquet" % cleaned_survey_path)
df_trips: pd.DataFrame = pd.read_parquet("%s/trips.parquet" % cleaned_survey_path)

In [139]:
df_trips


,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_cell,destination_cell,origin_activity_type,destination_activity_type,is_valid
0,0,0,0,pt,1110,900.0,38700.0,101001,102001,home,other,True
1,0,0,1,pt,1110,900.0,42300.0,102001,101001,other,home,True
2,0,1,2,pt,2590,1200.0,30000.0,101001,104001,home,work,True
3,0,1,3,pt,2750,1500.0,63900.0,104001,101001,work,home,True
4,0,2,4,pt,2390,600.0,26880.0,101001,212003,home,education,True
...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,3610,600.0,61200.0,712451,711006,home,leisure,True
99581,16359,36561,99581,car,3610,600.0,61800.0,711006,712451,leisure,home,True
99582,16360,36562,99582,car,770,300.0,27000.0,712451,712001,home,other,True
99583,16360,36562,99583,car,770,480.0,27420.0,712001,712451,other,work,True


In [140]:
df_trips = df_trips.merge(gdf_households[["household_id", "geometry"]], on="household_id")
df_trips.rename({
    "geometry": "household_geometry",
    "origin_cell": "origin_zone_id",
    "destination_cell": "destination_zone_id",
    "origin_activity": "origin_activity_id",
}, axis=1, inplace=True)

df_trips

,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,household_geometry
0,0,0,0,pt,1110,900.0,38700.0,101001,102001,home,other,True,POINT (841431.79 6517432.35)
1,0,0,1,pt,1110,900.0,42300.0,102001,101001,other,home,True,POINT (841431.79 6517432.35)
2,0,1,2,pt,2590,1200.0,30000.0,101001,104001,home,work,True,POINT (841431.79 6517432.35)
3,0,1,3,pt,2750,1500.0,63900.0,104001,101001,work,home,True,POINT (841431.79 6517432.35)
4,0,2,4,pt,2390,600.0,26880.0,101001,212003,home,education,True,POINT (841431.79 6517432.35)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99580,16359,36561,99580,car,3610,600.0,61200.0,712451,711006,home,leisure,True,POINT (849463.56 6522520.24)
99581,16359,36561,99581,car,3610,600.0,61800.0,711006,712451,leisure,home,True,POINT (849463.56 6522520.24)
99582,16360,36562,99582,car,770,300.0,27000.0,712451,712001,home,other,True,POINT (849874.93 6522224.13)
99583,16360,36562,99583,car,770,480.0,27420.0,712001,712451,other,work,True,POINT (849874.93 6522224.13)


In [141]:

if False:
    df_trips = df_trips.loc[:100].copy()


# Function to select a random point based on weight
def process_trip_group(trip_group):
    zone_id = trip_group.name[0]
    activity = trip_group.name[1]

    result = pd.Series([None] * len(trip_group), name="geometry")

    group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type != 'Point')]
    if group_zone.empty:
        group_zone = gdf_zones[(gdf_zones["zone_id"] == zone_id) & (gdf_zones["geometry"].geometry.type == 'Point')]
        if group_zone.empty:
            return result
        else:
            return  pd.Series([group_zone["geometry"].iloc[0]] * len(trip_group), name="geometry")

    group_zone_geometry = group_zone["geometry"].iloc[0]

    if not group_zone_geometry.is_valid:
        group_zone_geometry = make_valid(group_zone_geometry)
    
    if activity == "home":
        return trip_group["household_geometry"].rename("geometry")
    
    weights = None

    if activity == "work":
        gdf_filtered: gpd.GeoDataFrame = gdf_work[gdf_work.within(group_zone_geometry)]
        weights = "employees"
        if gdf_filtered.empty:
            gdf_filtered = gdf_work.iloc[[gdf_work.distance(group_zone_geometry).idxmin()]]
    elif activity == "education":
        gdf_filtered: gpd.GeoDataFrame = gdf_education[gdf_education.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_education.iloc[[gdf_education.distance(group_zone_geometry).idxmin()]]
        weights = "weight"
    elif activity in ["leisure", "shop", "other"]:
        mask = gdf_secondary.within(group_zone_geometry)
        mask &= gdf_secondary["activity_type"] == activity
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[mask]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]

    if gdf_filtered.empty:
        weights=None
        gdf_filtered: gpd.GeoDataFrame = gdf_secondary[gdf_secondary.within(group_zone_geometry)]
        if gdf_filtered.empty:
            gdf_filtered = gdf_secondary.iloc[[gdf_secondary.distance(group_zone_geometry).idxmin()]]
    
    result = gdf_filtered.sample(n=len(trip_group), weights=weights, replace=True)["geometry"].reset_index(drop=True)
    return result.rename("geometry")

# NEEEED to sort first !!!
tqdm.pandas(desc="Calculating origin_geometry")
df_trips.sort_values(["origin_zone_id", "origin_activity_type"], inplace=True)
origin_geometry = df_trips.groupby(["origin_zone_id", "origin_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["origin_geometry"] = origin_geometry.values

tqdm.pandas(desc="Calculating destination_geometry")
df_trips.sort_values(["destination_zone_id", "destination_activity_type"], inplace=True)
destination_geometry = df_trips.groupby(["destination_zone_id", "destination_activity_type"]).progress_apply(process_trip_group, include_groups=False).reset_index(drop=True)
df_trips["destination_geometry"] = destination_geometry.values

df_trips

Calculating origin_geometry:   0%|          | 0/7342 [00:00<?, ?it/s]

Calculating destination_geometry: 100%|██████████| 7277/7277 [07:08<00:00, 16.97it/s] 


,household_id,person_id,trip_id,mode,euclidean_distance,travel_time,departure_time,origin_zone_id,destination_zone_id,origin_activity_type,destination_activity_type,is_valid,household_geometry,origin_geometry,destination_geometry
80797,12383,27096,80797,walk,290,300.0,45000.0,101001,101001,leisure,education,True,POINT (842855.03 6518131.13),POINT (841426.77 6517319.71),POINT (841149.76 6517315.34)
91,19,33,91,walk,696,720.0,31500.0,101002,101001,home,education,True,POINT (841556.66 6518098.38),POINT (841556.66 6518098.38),POINT (841560.31 6517100.74)
93,19,33,93,walk,2609,2700.0,49500.0,101002,101001,home,education,True,POINT (841556.66 6518098.38),POINT (841556.66 6518098.38),POINT (841685.4 6517438)
20098,3330,6305,20098,pt,5720,1800.0,30000.0,136003,101001,home,education,True,POINT (846890.68 6519848.75),POINT (846890.68 6519848.75),POINT (841532.2 6516889)
33050,5443,10716,33050,pt,4240,1800.0,26100.0,214007,101001,home,education,True,POINT (840220.13 6516436.849),POINT (840220.130399673 6516436.849115813),POINT (841532.2 6516889)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167,165,309,1167,walk,116,120.0,50400.0,999090,999090,leisure,work,True,POINT (842242.75 6518641.21),None,None
97038,15810,35271,97038,car_passenger,0,300.0,47100.0,999090,999090,leisure,work,True,POINT (839499.14 6525074.6),None,None
80721,12369,27057,80721,car,0,720.0,64800.0,999100,999100,work,home,True,POINT (842997.31 6518669.96),None,None
57935,8530,18193,57935,walk,580,600.0,51600.0,999100,999100,home,work,True,POINT (846314.175 6503608.329),None,None


In [142]:

df_trips = df_trips[(~df_trips["origin_geometry"].isna()) & (~df_trips["destination_geometry"].isna())]

In [143]:
df_trips.loc[:, "trip_geometry"] = df_trips.apply(lambda x: geometry.LineString([x["origin_geometry"], x["destination_geometry"]]), axis=1)
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="trip_geometry", crs="EPSG:2154")
gdf_trips["household_geometry"] = gpd.GeoSeries(gdf_trips["household_geometry"]).to_wkt()
gdf_trips["origin_geometry"] = gpd.GeoSeries(gdf_trips["origin_geometry"]).to_wkt()
gdf_trips["destination_geometry"] = gpd.GeoSeries(gdf_trips["destination_geometry"]).to_wkt()

C:\Users\lebescond\AppData\Local\Temp\ipykernel_18244\4106265429.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_trips.loc[:, "trip_geometry"] = df_trips.apply(lambda x: geometry.LineString([x["origin_geometry"], x["destination_geometry"]]), axis=1)


In [144]:
gdf_trips.to_parquet("%s/trips.geoparquet" % cleaned_survey_path)